### dela upp markdown filen (det ska vara 40 olika punkter då exempelvis paragraf 6a, och 6b finns)

In [6]:
parts = []
with open("data/raw/kap12.md", encoding="utf-8") as f:
    for line in f:
        if line.startswith("## "):   # ny avdelning, sluta här
            break
        if line.startswith("**"):    # ny paragraf, starta en ny chunk
            parts.append("")
        if parts:                    # hoppa över allt före första paragrafen
            parts[-1] += line

assert len(parts) == 40, len(parts)

In [7]:
for i, p in enumerate(parts, start=1):
    print(f"{i:>2}: {p[:60].strip()!r}")

 1: '**1 §** Utgifter för att förvärva och bibehålla inkomster sk'
 2: '**2 §** Utgifter som avses i följande paragrafer ska dras av'
 3: '**3 §** Om den skattskyldige får sådan ersättning som enligt'
 4: '**4 §** Skattskyldiga som har sjöinkomst har inte rätt till'
 5: '**5 §** Utgifter för tjänsteresor med egen bil ska dras av m'
 6: '**6 §** Ökade levnadskostnader som den skattskyldige har på'
 7: '**6 a §** _/Upphör att gälla U:2027-01-01/_ När en riksdagsl'
 8: '**6 a §** _/Träder i kraft I:2027-01-01/_ När en riksdagsled'
 9: '**7 §** Med vanlig verksamhetsort avses ett område inom 50 k'
10: '**8 §** Tjänstestället är den plats där den skattskyldige ut'
11: '**9 §** Avdrag för ökade kostnader för logi, resor, måltider'
12: '**10 §** Vid beräkningen av avdraget används för tjänsteresa'
13: '**11 §** Maximibeloppet är 0,5 procent av prisbasbeloppet, a'
14: '**12 §** Om den anställde under en tjänsteresa samma dag upp'
15: '**13 §** Utgift för logi ska dras av. Kan den skattskyldige

### Förenkla schemat

In [8]:
from dotenv import load_dotenv
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

load_dotenv()  # läser COHERE_API_KEY från .env

embedding_model = get_registry().get(
    "cohere").create(name="embed-multilingual-v3.0")


# Test utan title osv 
class Paragraph(LanceModel):
    doc_id: str
    paragraph_nr: str
    content: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField()

### Gör rader av parts
- i är numereringen och p är parten 
- splita vid ny paragraf 
- append 

In [9]:
rows = []
for i, p in enumerate(parts, start=1):
    nr = p.split("§")[0].strip("* ")   # t.ex. "6 a"
    rows.append({
        "doc_id": f"12-{i}",           # löpnummer, så de två 6 a § blir unika
        "paragraph_nr": nr,
        "content": f"12 kap. {nr} § Inkomstskattelagen\n{p.strip()}",
    })

### test med LanceDB 
- skapar en vector databas med paragraferna som chunks och sök 
- add to the table och använd schema parapraph 

In [10]:
import lancedb

db = lancedb.connect("data/lancedb")
table = db.create_table("kap12", schema=Paragraph, mode="overwrite")
table.add(rows)   # här anropas Cohere och embeddingarna skapas

print(table.count_rows())  # ska bli 40

40


### Testa att söka 

In [13]:
results = table.search(
    "Hur mycket får jag dra av för resa med egen bil?").limit(3).to_pandas()

for _, r in results.iterrows():
    print(r["paragraph_nr"], r["_distance"])
    print(r["content"], "\n")

27 0.5543644428253174
12 kap. 27 § Inkomstskattelagen
**27 §** Utgifter för arbetsresor med egen bil ska dras av om avståndet mellan bostaden och arbetsplatsen är minst fem kilometer och det klart framgår att den skattskyldige genom att använda egen bil i stället för allmänna transportmedel regelmässigt gör en tidsvinst på sammanlagt minst två timmar.

Utgifter för arbetsresor med egen bil ska också dras av om bilen används i tjänsten minst 160 dagar under året. Om den används i tjänsten minst 60 dagar under året, ska utgifterna dras av för alla de dagar som bilen använts i tjänsten. Att bilen används i tjänsten beaktas dock bara om körsträckan är minst 300 mil per år.

Avdrag ska göras med 2 kronor och 50 öre för varje kilometer.

Utgifter för väg-, bro- och färjeavgifter samt skatt enligt lagen (2004:629) om trängselskatt ska också dras av.

_Lag (2022:1806)_. 

5 0.5852745175361633
12 kap. 5 § Inkomstskattelagen
**5 §** Utgifter för tjänsteresor med egen bil ska dras av med 2 kronor